In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error

physical_error_rate = 0.05

def generate_qiskit_dataset(num_shots=1000, physical_error_rate=0.05):
    print(f"Configuring Qiskit Noise Model ({physical_error_rate*100}% bit-flip risk)...")
    
    # 1. Build a Realistic Noise Model (FIXED)
    error_gate_1q = pauli_error([('X', physical_error_rate), ('I', 1 - physical_error_rate)])
    error_gate_2q = error_gate_1q.tensor(error_gate_1q) # Combines them for 2-qubit gates
    
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(error_gate_1q, ["id"])
    noise_model.add_all_qubit_quantum_error(error_gate_2q, ["cx"])

    # 2. Design the 3-Qubit Circuit
    qc = QuantumCircuit(5, 5)
    
    # --- Encoding Stage ---
    qc.cx(0, 1)
    qc.cx(0, 2)
    qc.barrier()
    
    # --- Syndrome Extraction Stage ---
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(1, 4)
    qc.cx(2, 4)
    qc.barrier()
    
    # --- Measurement Stage ---
    qc.measure(3, 0) # Syndrome bit 0
    qc.measure(4, 1) # Syndrome bit 1
    qc.measure(0, 2) # Data qubit 0 final state
    qc.measure(1, 3) # Data qubit 1 final state
    qc.measure(2, 4) # Data qubit 2 final state

    # 3. Execute with Memory Enabled
    simulator = AerSimulator(noise_model=noise_model)
    compiled_circuit = transpile(qc, simulator)
    
    result = simulator.run(compiled_circuit, shots=num_shots, memory=True).result()
    raw_shots = result.get_memory()

    # 4. Parse Raw Strings into ML Arrays (X and y)
    X = []
    y = []
    
    for shot in raw_shots:
        syndrome_bits = [int(shot[4]), int(shot[3])] 
        data_bits = [int(shot[2]), int(shot[1]), int(shot[0])]     
        
        logical_error = 1 if sum(data_bits) >= 2 else 0
        
        X.append(syndrome_bits)
        y.append(logical_error)
        
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

# Run the pipeline
X_qiskit, y_qiskit = generate_qiskit_dataset(num_shots=1000)
print(y_qiskit[:20])
print("\n--- QISKIT DATA EXTRACTION SUCCESSFUL ---")
print(f"Features (X) Matrix Shape: {X_qiskit.shape}")
print(f"Labels   (y) Vector Shape: {y_qiskit.shape}")
print("\nFirst 3 rows of your dataset:")
for i in range(3):
    print(f"Sample {i+1}: Features (X): {X_qiskit[i]} ---> Target (y): {y_qiskit[i]}")

Configuring Qiskit Noise Model (5.0% bit-flip risk)...
[[0. 0.]
 [1. 1.]
 [0. 0.]
 [1. 1.]
 [0. 0.]
 [1. 0.]
 [1. 1.]
 [0. 1.]
 [0. 0.]
 [1. 1.]
 [0. 1.]
 [0. 0.]
 [1. 0.]
 [0. 0.]
 [0. 0.]
 [1. 1.]
 [0. 0.]
 [0. 1.]
 [0. 0.]
 [0. 0.]]

--- QISKIT DATA EXTRACTION SUCCESSFUL ---
Features (X) Matrix Shape: (1000, 2)
Labels   (y) Vector Shape: (1000,)

First 3 rows of your dataset:
Sample 1: Features (X): [0. 0.] ---> Target (y): 0.0
Sample 2: Features (X): [1. 1.] ---> Target (y): 1.0
Sample 3: Features (X): [0. 0.] ---> Target (y): 0.0
